# RAG Ingest Pipeline on Kaggle: a heavy document vectorizer behind your Cloudflare Tunnel

Use a free Kaggle GPU to turn piles of documents into a searchable vector index, and take the vectors with you:

- **In:** PDF, DOCX, HTML, Markdown, text, CSV/JSON, source code and notebooks, from uploads, URLs, raw text, or whole folders attached as a Kaggle dataset (thousands of files).
- **Processing:** text extraction with page numbers and headings → overlapping chunks → embeddings on the GPU with [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) (multilingual, 100+ languages, 1024 dimensions). Duplicate files are skipped.
- **Search:** exact semantic search, optionally reranked with [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3), from the web page, the API or an agent (MCP).
- **Out:** a zip per collection with `chunks.jsonl` + `embeddings.npy` (+ `documents.jsonl`, `manifest.json`), ready to load into Qdrant, pgvector, Chroma, LanceDB, FAISS...

| Path | What |
|---|---|
| `/` | Collections, ingest, search and export page |
| `/api/*` | JSON API |
| `/mcp` | MCP server for agents (tools: `list_collections`, `create_collection`, `ingest`, `job_status`, `search`, `get_chunks`, `list_documents`, `delete_document`, `export_collection`, `list_input_files`, `server_status`) |

### Before you run
1. **Accelerator:** GPU T4 x2 (ingestion embeds on GPU 0 while searches and reranking use GPU 1). One GPU or even CPU works, more slowly. **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `RAG_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `RAG_UI_PASSWORD` (recommended): browser login is any username + this password; agents send `Authorization: Bearer <password>`. Without it a random password is printed below.
4. **Big corpora:** attach them as a Kaggle dataset (Add Input) and ingest the folder from the page or with the `ingest` tool, instead of uploading through Cloudflare (100 MB per request).
5. **Run All.** First run: about 4 minutes (install + 4.5 GB of models).

Scanned PDFs without a text layer need OCR first; the page reports which pages had no text. Collections live in `/kaggle/working/rag/collections` and disappear when the session ends: export what you want to keep. Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings
EMBED_MODEL = "BAAI/bge-m3"                    # any sentence-transformers model; e.g. "intfloat/multilingual-e5-large"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"       # any cross-encoder
QUERY_PREFIX, DOC_PREFIX = "", ""              # e5 models want "query: " and "passage: "
EMBED_BATCH = 32                               # lower it if you see CUDA out-of-memory errors in the log
PORT = 7860
WORK_DIR = "/kaggle/working/rag"
LOG_DIR = "/kaggle/working/logs"
APP_DIR = '/kaggle/working/rag_app'
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers get a login page and a signed session cookie (no browser pop-up);
  agents send ``Authorization: Bearer <password>``, ``X-Access-Token: <password>`` or HTTP Basic.
  ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hashlib
import hmac
import html as _html
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
SESSION_COOKIE = "studio_session"
SESSION_DAYS = 30
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _strip_session(cookie):
    """Never forward our session cookie to a proxied backend."""
    return "; ".join(p.strip() for p in (cookie or "").split(";")
                     if p.strip() and p.strip().partition("=")[0] != SESSION_COOKIE)


LOGIN_PAGE = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Sign in · {{NAME}}</title>
<style>
:root{--bg:#f4f5f8;--card:#fff;--text:#14171c;--muted:#636b77;--line:#dfe2e8;--accent:#2f6fec;--bad:#c4372f}
@media (prefers-color-scheme:dark){:root{--bg:#0d0f13;--card:#161a20;--text:#e8eaee;--muted:#98a0ab;--line:#2a2f37;--accent:#5b8ff5;--bad:#ef6a60}}
*{box-sizing:border-box}html,body{height:100%;margin:0}
body{background:radial-gradient(1200px 600px at 10% -10%,color-mix(in srgb,var(--accent) 18%,transparent),transparent),var(--bg);
color:var(--text);font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,"Noto Sans","Noto Sans Devanagari",sans-serif;
display:flex;align-items:center;justify-content:center;padding:16px}
.card{width:100%;max-width:380px;background:var(--card);border:1px solid var(--line);border-radius:18px;padding:30px 28px;
box-shadow:0 20px 60px rgba(0,0,0,.12)}
.logo{width:44px;height:44px;border-radius:12px;background:var(--accent);display:flex;align-items:center;justify-content:center;
color:#fff;font-weight:800;font-size:20px;margin-bottom:18px}
h1{font-size:20px;margin:0 0 4px}p{color:var(--muted);margin:0 0 22px;font-size:14px}
label{display:block;font-size:13px;color:var(--muted);margin-bottom:6px}
.field{position:relative}
input{width:100%;font:inherit;color:var(--text);background:transparent;border:1px solid var(--line);border-radius:10px;padding:11px 44px 11px 12px}
input:focus{outline:2px solid color-mix(in srgb,var(--accent) 45%,transparent);border-color:var(--accent)}
.eye{position:absolute;right:6px;top:50%;transform:translateY(-50%);border:0;background:none;color:var(--muted);cursor:pointer;padding:6px;font-size:13px}
button.go{width:100%;margin-top:16px;border:0;border-radius:10px;padding:12px;background:var(--accent);color:#fff;font:600 15px system-ui;cursor:pointer}
button.go:disabled{opacity:.6}.err{color:var(--bad);font-size:13px;min-height:20px;margin-top:10px}
.hint{font-size:12px;color:var(--muted);margin-top:18px;border-top:1px solid var(--line);padding-top:14px}
</style></head><body>
<form class="card" id="f" method="post" action="/login">
<div class="logo">&#9656;</div><h1>{{NAME}}</h1><p>Sign in with the password from your notebook.</p>
<label for="pw">Password</label><div class="field"><input id="pw" name="password" type="password" autocomplete="current-password" autofocus required>
<button class="eye" type="button" id="eye">Show</button></div>
<button class="go" id="go">Sign in</button><div class="err" id="err"></div>
<div class="hint">It is the <b>*_UI_PASSWORD</b> Kaggle secret, or the password the notebook printed.</div>
</form>
<script>
const f=document.getElementById('f'),pw=document.getElementById('pw'),err=document.getElementById('err'),go=document.getElementById('go');
document.getElementById('eye').onclick=e=>{pw.type=pw.type==='password'?'text':'password';e.target.textContent=pw.type==='password'?'Show':'Hide';};
f.onsubmit=async e=>{e.preventDefault();go.disabled=true;err.textContent='';
 try{const r=await fetch('/login',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({password:pw.value})});
  if(r.ok){location.reload();return;} const d=await r.json().catch(()=>({}));err.textContent=d.error||('Sign-in failed ('+r.status+')');}
 catch(x){err.textContent='Network error: '+x;} go.disabled=false;pw.select();};
</script></body></html>"""


def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []
        self._fails = {}

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/login", auth=False)(self._login)
        self.route("GET", "/logout", auth=False)(self._logout)
        self.route("POST", "/logout", auth=False)(self._logout)
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def _key(self):
        return hashlib.sha256(("studio-session:" + self.password).encode("utf-8")).digest()

    def make_session(self, days=SESSION_DAYS):
        exp = str(int(time.time() + days * 86400))
        return "%s.%s" % (exp, hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40])

    def _session_ok(self, headers):
        for part in (headers.get("Cookie") or "").split(";"):
            k, _, v = part.strip().partition("=")
            if k == SESSION_COOKIE:
                exp, _, sig = v.partition(".")
                if exp.isdigit() and int(exp) > time.time():
                    want = hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40]
                    if hmac.compare_digest(sig.encode(), want.encode()):
                        return True
        return False

    def _cookie(self, headers, value, max_age):
        secure = (headers.get("X-Forwarded-Proto") or "").lower() == "https" or '"https"' in (headers.get("Cf-Visitor") or "")
        return "%s=%s; Path=/; Max-Age=%d; HttpOnly; SameSite=Lax%s" % (
            SESSION_COOKIE, value, max_age, "; Secure" if secure else "")

    def _login(self, req):
        if not self.password:
            return {"ok": True}
        ip = req.headers.get("Cf-Connecting-Ip") or req.handler.client_address[0]
        now = time.time()
        fails = [t for t in self._fails.get(ip, []) if now - t < 300]
        if len(fails) >= 8:
            raise HTTPError(429, "too many attempts; wait a few minutes")
        if "json" in (req.headers.get("Content-Type") or ""):
            given = str(req.json().get("password") or "")
        else:
            given = parse_qs(req.body().decode("utf-8", "replace")).get("password", [""])[0]
        if not hmac.compare_digest(given.encode("utf-8"), self.password.encode("utf-8")):
            fails.append(now)
            self._fails[ip] = fails
            time.sleep(0.5)
            raise HTTPError(401, "wrong password")
        self._fails.pop(ip, None)
        return Response(json.dumps({"ok": True}), 200, "application/json; charset=utf-8",
                        {"Set-Cookie": self._cookie(req.headers, self.make_session(), SESSION_DAYS * 86400)})

    def _logout(self, req):
        return Response(b"", 303, "text/plain", {"Location": "/", "Set-Cookie": self._cookie(req.headers, "", 0)})

    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True
        if self._session_ok(headers):
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        """Pages get the login form; everything else a JSON 401 (no browser pop-up)."""
        if h.command in ("GET", "HEAD") and "text/html" in (h.headers.get("Accept") or ""):
            return self._respond(h, Response(LOGIN_PAGE.replace("{{NAME}}", _html.escape(self.name)), 401,
                                             "text/html; charset=utf-8"))
        self._respond(h, Response(json.dumps({"error": "login required: sign in on the page, or send "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8"))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers = [(k, v) for k, v in headers if v]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                  if k.lower() not in SECRET_HEADERS and (k.lower() != "cookie" or _strip_session(v))]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  if (r.status === 401) { location.reload(); throw new Error("Signed out; please sign in again"); }
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Parsing (PDF, DOCX, HTML, Markdown, text, code) and chunking
# Writes rag_parse.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'rag_parse.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Document parsing and chunking for the RAG ingest pipeline (no GPU libraries).

parse_file() turns a file into blocks: [{"text", "page", "heading"}] in reading order.
chunk_blocks() packs blocks into overlapping chunks that keep their page range and heading.
PDF needs pymupdf, DOCX needs python-docx; everything else is the standard library.
"""
import html.parser
import json
import os
import re

KINDS = {".pdf": "pdf", ".docx": "docx", ".html": "html", ".htm": "html", ".xhtml": "html",
         ".md": "markdown", ".markdown": "markdown", ".mdx": "markdown", ".txt": "text", ".rst": "text",
         ".csv": "text", ".tsv": "text", ".json": "text", ".jsonl": "text", ".xml": "text", ".log": "text",
         ".ipynb": "notebook", ".py": "code", ".js": "code", ".ts": "code", ".tsx": "code", ".jsx": "code",
         ".java": "code", ".go": "code", ".rs": "code", ".c": "code", ".h": "code", ".cpp": "code", ".hpp": "code",
         ".cs": "code", ".rb": "code", ".php": "code", ".kt": "code", ".swift": "code", ".scala": "code",
         ".sql": "code", ".sh": "code", ".yaml": "code", ".yml": "code", ".toml": "code", ".ini": "code"}
SUPPORTED = tuple(KINDS)


def kind_of(name, content_type=""):
    ext = os.path.splitext(name.lower().split("?")[0])[1]
    if ext in KINDS:
        return KINDS[ext]
    ct = (content_type or "").lower()
    for key, kind in (("pdf", "pdf"), ("html", "html"), ("wordprocessingml", "docx"), ("markdown", "markdown"),
                      ("json", "text"), ("text/", "text")):
        if key in ct:
            return kind
    return None


def _paras(text, page=None, heading=None, markdown=False):
    blocks = []
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if not para:
            continue
        if markdown:
            m = re.match(r"^(#{1,6})\s+(.+)$", para.splitlines()[0])
            if m:
                heading = m.group(2).strip()
        blocks.append({"text": para, "page": page, "heading": heading})
    return blocks, heading


# ---------------------------------------------------------------------- formats
def _pdf(path):
    import pymupdf
    doc = pymupdf.open(path)
    blocks, empty = [], 0
    title = (doc.metadata or {}).get("title") or ""
    for i, page in enumerate(doc, 1):
        text = page.get_text("text")
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # join hyphenated line breaks
        text = re.sub(r"(?<![\n.!?:])\n(?!\n)", " ", text)     # unwrap lines inside paragraphs
        if not text.strip():
            empty += 1
            continue
        b, _ = _paras(text, page=i)
        blocks += b
    pages = len(doc)
    doc.close()
    warn = ["%d of %d pages have no text layer (scanned?); run OCR first to include them" % (empty, pages)] if empty else []
    return title, blocks, pages, warn


def _docx(path):
    import docx
    from docx.table import Table
    from docx.text.paragraph import Paragraph
    d = docx.Document(path)
    blocks, heading = [], None
    title = d.core_properties.title or ""
    for child in d.element.body.iterchildren():
        tag = child.tag.rsplit("}", 1)[-1]
        if tag == "p":
            p = Paragraph(child, d)
            text = p.text.strip()
            if not text:
                continue
            style = (p.style.name if p.style is not None else "") or ""
            if style.lower().startswith(("heading", "title")):
                heading = text
                title = title or (text if style.lower() == "title" else "")
            blocks.append({"text": text, "page": None, "heading": heading})
        elif tag == "tbl":
            rows = []
            for row in Table(child, d).rows:
                cells = []
                for c in row.cells:
                    t = " ".join(c.text.split())
                    if not cells or cells[-1] != t:                    # merged cells repeat
                        cells.append(t)
                rows.append(" | ".join(cells))
            if rows:
                blocks.append({"text": "\n".join(rows), "page": None, "heading": heading})
    return title, blocks, None, []


class _HTML(html.parser.HTMLParser):
    SKIP = {"script", "style", "noscript", "svg", "template", "iframe", "head"}
    BLOCK = {"p", "div", "section", "article", "li", "tr", "br", "h1", "h2", "h3", "h4", "h5", "h6", "pre",
             "blockquote", "td", "th", "dd", "dt", "table", "ul", "ol", "main", "header", "footer", "nav", "aside"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.blocks, self.buf, self.skip, self.title, self.in_title, self.heading, self.hbuf = [], [], 0, "", False, None, None

    def flush(self):
        text = re.sub(r"[ \t\r\f\v]+", " ", "".join(self.buf)).strip()
        self.buf = []
        if text:
            self.blocks.append({"text": text, "page": None, "heading": self.heading})

    def handle_starttag(self, tag, attrs):
        if tag == "title":
            self.in_title = True
        elif tag in self.SKIP:
            self.skip += 1
        elif tag in self.BLOCK:
            self.flush()
            if re.fullmatch(r"h[1-6]", tag):
                self.hbuf = []

    def handle_endtag(self, tag):
        if tag == "title":
            self.in_title = False
        elif tag in self.SKIP:
            self.skip = max(0, self.skip - 1)
        elif tag in self.BLOCK:
            if re.fullmatch(r"h[1-6]", tag) and self.hbuf is not None:
                self.heading = " ".join("".join(self.hbuf).split()) or self.heading
                self.hbuf = None
            self.flush()

    def handle_data(self, data):
        if self.in_title:
            self.title += data
        elif not self.skip:
            self.buf.append(data)
            if self.hbuf is not None:
                self.hbuf.append(data)


def parse_html(text):
    p = _HTML()
    p.feed(text)
    p.close()
    p.flush()
    # glue tiny fragments (menu items, table cells) into paragraphs
    blocks = []
    for b in p.blocks:
        if blocks and len(blocks[-1]["text"]) < 200 and blocks[-1]["heading"] == b["heading"]:
            blocks[-1]["text"] += "\n" + b["text"]
        else:
            blocks.append(b)
    return " ".join(p.title.split()), blocks


def _notebook(text):
    nb = json.loads(text)
    blocks, heading = [], None
    for c in nb.get("cells", []):
        src = "".join(c.get("source", [])) if isinstance(c.get("source"), list) else c.get("source", "")
        if c.get("cell_type") == "markdown":
            b, heading = _paras(src, heading=heading, markdown=True)
            blocks += b
        elif src.strip():
            blocks.append({"text": src.strip(), "page": None, "heading": heading})
    return blocks


def read_text(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return raw.decode("utf-8", "replace")


def parse_file(path, name=None, content_type=""):
    """Returns {"title", "kind", "blocks", "pages", "warnings"}."""
    name = name or os.path.basename(path)
    kind = kind_of(name, content_type)
    if kind is None:
        raise ValueError("unsupported file type: %s" % name)
    title, pages, warn = "", None, []
    if kind == "pdf":
        title, blocks, pages, warn = _pdf(path)
    elif kind == "docx":
        title, blocks, pages, warn = _docx(path)
    elif kind == "html":
        title, blocks = parse_html(read_text(path))
    elif kind == "notebook":
        blocks = _notebook(read_text(path))
    elif kind == "code":
        blocks, _ = _paras(read_text(path), heading=name)
    else:
        blocks, _ = _paras(read_text(path), markdown=(kind == "markdown"))
    if kind == "markdown" and not title:
        m = re.search(r"^#\s+(.+)$", read_text(path), re.M)
        title = m.group(1).strip() if m else ""
    return {"title": (title or os.path.splitext(name)[0]).strip()[:200], "kind": kind, "blocks": blocks,
            "pages": pages, "warnings": warn}


# ---------------------------------------------------------------------- chunking
def _pieces(text, size):
    if len(text) <= size:
        return [text]
    out, cur = [], ""
    for sent in re.split(r"(?<=[.!?。！？])\s+|\n", text):
        while len(sent) > size:                                   # a single huge "sentence"
            cut = sent.rfind(" ", 0, size)
            cut = cut if cut > size // 2 else size
            out.append((cur + " " + sent[:cut]).strip() if cur else sent[:cut])
            cur, sent = "", sent[cut:].strip()
        if len(cur) + len(sent) + 1 > size and cur:
            out.append(cur)
            cur = sent
        else:
            cur = (cur + " " + sent).strip()
    if cur:
        out.append(cur)
    return out


def _tail(text, n):
    if n <= 0 or len(text) <= n:
        return text if n > 0 else ""
    t = text[-n:]
    sp = t.find(" ")
    return t[sp + 1:] if 0 <= sp < n // 2 else t


def chunk_blocks(blocks, size=1200, overlap=150):
    """Packs blocks into chunks of about ``size`` characters, overlapping by ``overlap``."""
    size, overlap = max(200, int(size)), max(0, min(int(overlap), int(size) // 2))
    chunks, cur = [], None

    def emit():
        if cur and cur["text"].strip():
            chunks.append(dict(cur, text=cur["text"].strip()))
    for b in blocks:
        for piece in _pieces(b["text"], size - overlap if overlap else size):
            if cur and len(cur["text"]) + len(piece) + 2 <= size and cur["heading"] == b["heading"]:
                cur["text"] += "\n\n" + piece
                if b["page"] is not None:
                    cur["page_end"] = b["page"]
                    cur["page_start"] = cur["page_start"] or b["page"]
                continue
            emit()
            carry = _tail(cur["text"], overlap) if cur and cur["heading"] == b["heading"] else ""
            cur = {"text": (carry + "\n\n" + piece) if carry else piece, "heading": b["heading"],
                   "page_start": b["page"], "page_end": b["page"]}
    emit()
    return chunks


def embed_text(chunk, title):
    """What gets embedded: the chunk with its document title and heading as context."""
    ctx = " - ".join(x for x in (title, chunk.get("heading")) if x)
    return (ctx + "\n" + chunk["text"]) if ctx else chunk["text"]
''')
print("wrote", os.path.join(APP_DIR, 'rag_parse.py'))

In [ ]:
# Collections: SQLite + float16 vectors, exact search, export
# Writes rag_store.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'rag_store.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Collections for the RAG ingest pipeline: SQLite for documents and chunk text, an append-only
float16 matrix for the vectors (row i = chunk id i). No vector-database dependency.

Search is exact (a matrix product over every live vector) on the GPU when torch + CUDA are there,
else on the CPU with numpy. That stays fast into the low millions of chunks on a T4.
"""
import json
import os
import re
import shutil
import sqlite3
import threading
import time

import numpy as np

NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_-]{0,39}$")
SCHEMA = """
CREATE TABLE IF NOT EXISTS documents(id INTEGER PRIMARY KEY AUTOINCREMENT, source TEXT, title TEXT, kind TEXT,
    sha256 TEXT, pages INTEGER, chunks INTEGER, chars INTEGER, added REAL, meta TEXT, deleted INTEGER DEFAULT 0);
CREATE INDEX IF NOT EXISTS documents_sha ON documents(sha256);
CREATE TABLE IF NOT EXISTS chunks(id INTEGER PRIMARY KEY, doc_id INTEGER, ord INTEGER, text TEXT,
    page_start INTEGER, page_end INTEGER, heading TEXT, deleted INTEGER DEFAULT 0);
CREATE INDEX IF NOT EXISTS chunks_doc ON chunks(doc_id);
"""


class Collection:
    def __init__(self, root, name, model=None, dim=None, chunk_size=1200, chunk_overlap=150, create=False):
        if not NAME.match(name or ""):
            raise ValueError("collection names use letters, digits, - and _ (max 40)")
        self.name, self.dir = name, os.path.join(root, name)
        self.lock = threading.RLock()
        meta_path = os.path.join(self.dir, "meta.json")
        if not os.path.exists(meta_path):
            if not create:
                raise KeyError("no collection named %r" % name)
            os.makedirs(self.dir, exist_ok=True)
            self._write_meta({"name": name, "model": model, "dim": dim, "chunk_size": int(chunk_size),
                              "chunk_overlap": int(chunk_overlap), "created": time.time()})
        self.meta = json.load(open(meta_path))
        self.db = sqlite3.connect(os.path.join(self.dir, "chunks.sqlite"), check_same_thread=False)
        self.db.executescript(SCHEMA)
        self.vec_path = os.path.join(self.dir, "vectors.f16")
        self.version = 0
        self._repair()

    def _write_meta(self, meta):
        tmp = os.path.join(self.dir, "meta.json.tmp")
        with open(tmp, "w") as f:
            json.dump(meta, f, indent=1)
        os.replace(tmp, os.path.join(self.dir, "meta.json"))

    # ------------------------------------------------------------------ vectors
    @property
    def dim(self):
        return self.meta.get("dim")

    def rows(self):
        if not self.dim or not os.path.exists(self.vec_path):
            return 0
        return os.path.getsize(self.vec_path) // (self.dim * 2)

    def _repair(self):
        """After a crash between writing vectors and committing rows, drop the extra vectors."""
        with self.lock:
            n_db = self.db.execute("SELECT COALESCE(MAX(id) + 1, 0) FROM chunks").fetchone()[0]
            if self.dim and self.rows() > n_db:
                with open(self.vec_path, "r+b") as f:
                    f.truncate(n_db * self.dim * 2)

    def vectors(self):
        n = self.rows()
        if not n:
            return np.zeros((0, self.dim or 1), dtype=np.float16)
        return np.memmap(self.vec_path, dtype=np.float16, mode="r", shape=(n, self.dim))

    # ------------------------------------------------------------------ documents
    def find_sha(self, sha):
        r = self.db.execute("SELECT id FROM documents WHERE sha256=? AND deleted=0", (sha,)).fetchone()
        return r[0] if r else None

    def add_document(self, doc, chunks, vecs):
        """doc: {source, title, kind, sha256, pages, meta}; chunks: [{text, heading, page_start, page_end}];
        vecs: float array (len(chunks), dim), already L2-normalised."""
        vecs = np.ascontiguousarray(vecs, dtype=np.float16)
        if len(chunks) != len(vecs):
            raise ValueError("chunks and vectors differ in length")
        with self.lock:
            if not self.dim:
                self.meta["dim"] = int(vecs.shape[1])
                self._write_meta(self.meta)
            if vecs.shape[1] != self.dim:
                raise ValueError("vector size %d does not match this collection (%d); it was built with %s"
                                 % (vecs.shape[1], self.dim, self.meta.get("model")))
            start = self.db.execute("SELECT COALESCE(MAX(id) + 1, 0) FROM chunks").fetchone()[0]
            if self.rows() != start:
                self._repair()
            with open(self.vec_path, "ab") as f:
                f.write(vecs.tobytes())
            cur = self.db.execute(
                "INSERT INTO documents(source, title, kind, sha256, pages, chunks, chars, added, meta) "
                "VALUES(?,?,?,?,?,?,?,?,?)",
                (doc.get("source"), doc.get("title"), doc.get("kind"), doc.get("sha256"), doc.get("pages"),
                 len(chunks), sum(len(c["text"]) for c in chunks), time.time(), json.dumps(doc.get("meta") or {})))
            doc_id = cur.lastrowid
            self.db.executemany(
                "INSERT INTO chunks(id, doc_id, ord, text, page_start, page_end, heading) VALUES(?,?,?,?,?,?,?)",
                [(start + i, doc_id, i, c["text"], c.get("page_start"), c.get("page_end"), c.get("heading"))
                 for i, c in enumerate(chunks)])
            self.db.commit()
            self.version += 1
            return doc_id

    def delete_document(self, doc_id):
        with self.lock:
            n = self.db.execute("UPDATE documents SET deleted=1 WHERE id=? AND deleted=0", (int(doc_id),)).rowcount
            if not n:
                raise KeyError("no document %s in %s" % (doc_id, self.name))
            self.db.execute("UPDATE chunks SET deleted=1 WHERE doc_id=?", (int(doc_id),))
            self.db.commit()
            self.version += 1
            return {"deleted": int(doc_id)}

    def documents(self, offset=0, limit=50, q=""):
        where, args = "deleted=0", []
        if q:
            where += " AND (title LIKE ? OR source LIKE ?)"
            args += ["%" + q + "%"] * 2
        total = self.db.execute("SELECT COUNT(*) FROM documents WHERE " + where, args).fetchone()[0]
        rows = self.db.execute("SELECT id, source, title, kind, pages, chunks, chars, added, meta FROM documents WHERE "
                               + where + " ORDER BY id DESC LIMIT ? OFFSET ?", args + [int(limit), int(offset)])
        keys = ("id", "source", "title", "kind", "pages", "chunks", "chars", "added", "meta")
        docs = []
        for r in rows:
            d = dict(zip(keys, r))
            d["meta"] = json.loads(d["meta"] or "{}")
            docs.append(d)
        return {"total": total, "documents": docs}

    def document(self, doc_id):
        d = self.db.execute("SELECT id, source, title, kind, pages, chunks, chars, added FROM documents "
                            "WHERE id=? AND deleted=0", (int(doc_id),)).fetchone()
        if not d:
            raise KeyError("no document %s in %s" % (doc_id, self.name))
        out = dict(zip(("id", "source", "title", "kind", "pages", "chunks", "chars", "added"), d))
        out["chunk_list"] = self.chunks_of(doc_id)
        return out

    def chunks_of(self, doc_id):
        return [self._chunk(r) for r in self.db.execute(
            "SELECT c.id, c.doc_id, c.ord, c.text, c.page_start, c.page_end, c.heading, d.title, d.source "
            "FROM chunks c JOIN documents d ON d.id=c.doc_id WHERE c.doc_id=? AND c.deleted=0 ORDER BY c.ord",
            (int(doc_id),))]

    @staticmethod
    def _chunk(r):
        return {"chunk_id": r[0], "document_id": r[1], "ord": r[2], "text": r[3], "page_start": r[4],
                "page_end": r[5], "heading": r[6], "title": r[7], "source": r[8]}

    def get_chunks(self, ids, context=0):
        """Chunks by id; context=N also returns N neighbours on each side within the same document."""
        ids = [int(i) for i in ids][:100]
        context = max(0, min(5, int(context or 0)))
        out = []
        for cid in ids:
            r = self.db.execute("SELECT doc_id, ord FROM chunks WHERE id=? AND deleted=0", (cid,)).fetchone()
            if not r:
                continue
            rows = self.db.execute(
                "SELECT c.id, c.doc_id, c.ord, c.text, c.page_start, c.page_end, c.heading, d.title, d.source "
                "FROM chunks c JOIN documents d ON d.id=c.doc_id WHERE c.doc_id=? AND c.ord BETWEEN ? AND ? "
                "AND c.deleted=0 ORDER BY c.ord", (r[0], r[1] - context, r[1] + context))
            out.append({"chunk_id": cid, "chunks": [self._chunk(x) for x in rows]})
        return out

    def stats(self):
        docs, chunks, chars = self.db.execute(
            "SELECT COUNT(*), COALESCE(SUM(chunks),0), COALESCE(SUM(chars),0) FROM documents WHERE deleted=0").fetchone()
        return dict(self.meta, documents=docs, chunks=chunks, chars=chars,
                    disk_bytes=sum(os.path.getsize(os.path.join(self.dir, f)) for f in os.listdir(self.dir)))

    def live_mask(self):
        n = self.rows()
        mask = np.zeros(n, dtype=bool)
        ids = [r[0] for r in self.db.execute("SELECT id FROM chunks WHERE deleted=0")]
        ids = [i for i in ids if i < n]
        mask[ids] = True
        return mask

    # ------------------------------------------------------------------ export
    def export_files(self):
        """(path_or_bytes, arcname) pairs for a zip: live chunks, their vectors and documents."""
        with self.lock:
            mask = self.live_mask()
            idx = np.nonzero(mask)[0]
            emb = os.path.join(self.dir, "export_embeddings.npy")
            np.save(emb, np.asarray(self.vectors()[idx]) if len(idx) else np.zeros((0, self.dim or 0), np.float16))
            lines = []
            for i, cid in enumerate(idx.tolist()):
                r = self.db.execute(
                    "SELECT c.id, c.doc_id, c.ord, c.text, c.page_start, c.page_end, c.heading, d.title, d.source "
                    "FROM chunks c JOIN documents d ON d.id=c.doc_id WHERE c.id=?", (cid,)).fetchone()
                lines.append(json.dumps(dict(self._chunk(r), row=i), ensure_ascii=False))
            docs = self.documents(0, 10 ** 9)["documents"]
        a = self.name
        manifest = dict(self.stats(), exported=time.time(), files={
            "chunks.jsonl": "one chunk per line; 'row' is its row in embeddings.npy",
            "embeddings.npy": "float16 matrix (rows, dim), L2-normalised: cosine similarity = dot product",
            "documents.jsonl": "one source document per line"})
        return [(json.dumps(manifest, indent=1).encode(), a + "/manifest.json"),
                (("\n".join(lines) + "\n").encode(), a + "/chunks.jsonl"),
                (emb, a + "/embeddings.npy"),
                (("\n".join(json.dumps(d, ensure_ascii=False) for d in docs) + "\n").encode(), a + "/documents.jsonl")]

    def close(self):
        self.db.close()

    def destroy(self):
        self.close()
        shutil.rmtree(self.dir)


class Searcher:
    """Keeps each collection's live vectors in memory (GPU when possible) and runs exact search."""

    def __init__(self, device=None):
        self.device = device
        self.cache = {}

    def _matrix(self, col):
        key = col.name
        hit = self.cache.get(key)
        if hit and hit[0] == (col.version, col.rows()):
            return hit[1], hit[2]
        with col.lock:
            mask = col.live_mask()
            idx = np.nonzero(mask)[0]
            vecs = np.asarray(col.vectors()[idx], dtype=np.float16)
            stamp = (col.version, col.rows())
        mat = vecs
        if self.device and self.device.startswith("cuda"):
            import torch
            mat = torch.from_numpy(vecs).to(self.device)
        else:
            mat = vecs.astype(np.float32)
        self.cache[key] = (stamp, mat, idx)
        return mat, idx

    def search(self, col, qvec, k, allowed=None):
        mat, idx = self._matrix(col)
        if not len(idx):
            return []
        if self.device and self.device.startswith("cuda"):
            import torch
            q = torch.from_numpy(np.asarray(qvec, dtype=np.float16)).to(self.device)
            scores = (mat @ q).float().cpu().numpy()
        else:
            scores = mat @ np.asarray(qvec, dtype=np.float32)
        if allowed is not None:
            scores = np.where(np.isin(idx, allowed), scores, -np.inf)
        k = min(int(k), len(idx))
        top = np.argpartition(-scores, k - 1)[:k]
        top = top[np.argsort(-scores[top])]
        return [(int(idx[t]), float(scores[t])) for t in top if np.isfinite(scores[t])]

    def forget(self, name):
        self.cache.pop(name, None)
''')
print("wrote", os.path.join(APP_DIR, 'rag_store.py'))

In [ ]:
# Server on port 7860: ingest queue, search, API and MCP
# Writes rag_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'rag_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""RAG Ingest Pipeline: a heavy document vectorizer on port 7860.

Runs as its own process (started by the notebook with studio_http.spawn):
    APP_PASSWORD=... python rag_server.py --port 7860

Files, folders, URLs and raw text -> parsed (PDF, DOCX, HTML, Markdown, text, code, notebooks) ->
chunked with page and heading metadata -> embedded on the GPU (BAAI/bge-m3 by default) -> stored per
collection -> searchable (with optional reranking) and exportable as chunks.jsonl + embeddings.npy
for any vector database. With two GPUs, ingestion embeds on GPU 0 while searches use GPU 1.
"""
import argparse
import hashlib
import os
import re
import shutil
import threading
import time
import traceback
import urllib.request
import uuid

import rag_parse as P
import rag_store as R
import studio_http as K

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("WORK_DIR", "/kaggle/working/rag")
COLS_DIR = os.path.join(WORK, "collections")
UPLOADS = os.path.join(WORK, "uploads")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "BAAI/bge-m3")
RERANK_MODEL = os.environ.get("RERANK_MODEL", "BAAI/bge-reranker-v2-m3")
QUERY_PREFIX = os.environ.get("QUERY_PREFIX", "")
DOC_PREFIX = os.environ.get("DOC_PREFIX", "")
MAX_SEQ = int(os.environ.get("EMBED_MAX_TOKENS", "1024"))
BATCH = int(os.environ.get("EMBED_BATCH", "32"))
INPUT_ROOTS = [r for r in ("/kaggle/input", "/kaggle/working") if os.path.isdir(r)] or [WORK]
MAX_FETCH = 200 << 20
MAX_WAIT = 85
log = K.file_logger(os.environ.get("APP_LOG", "/kaggle/working/logs/rag.log"), "rag")

LOCK = threading.RLock()
WAKE = threading.Condition(LOCK)
COLS = {}
JOBS = {}               # id -> job (newest last)
QUEUE = []
M = {"devices": None, "ingest": None, "query": None, "rerank": None, "searcher": None}


# ====================================================================== models
def devices():
    if M["devices"] is None:
        try:
            import torch
            n = torch.cuda.device_count()
        except Exception:
            n = 0
        M["devices"] = {"n": n, "ingest": "cuda:0" if n else "cpu",
                        "query": "cuda:1" if n > 1 else ("cuda:0" if n else "cpu")}
    return M["devices"]


class Embedder:
    def __init__(self, device):
        from sentence_transformers import SentenceTransformer
        self.device = device
        self.model = SentenceTransformer(EMBED_MODEL, device=device, trust_remote_code=False)
        if device.startswith("cuda"):
            self.model.half()
        self.model.max_seq_length = min(MAX_SEQ, self.model.max_seq_length or MAX_SEQ)
        self.dim = self.model.get_sentence_embedding_dimension()
        self.lock = threading.Lock()
        log.info("loaded %s on %s (dim %d)", EMBED_MODEL, device, self.dim)

    def encode(self, texts, prefix=""):
        with self.lock:
            return self.model.encode([prefix + t for t in texts], batch_size=BATCH, normalize_embeddings=True,
                                     convert_to_numpy=True, show_progress_bar=False).astype("float16")


def embedder(role):
    with LOCK:
        if M[role] is None:
            dv = devices()
            if role == "query" and dv["query"] == dv["ingest"] and M["ingest"] is not None:
                M["query"] = M["ingest"]
            elif role == "ingest" and dv["query"] == dv["ingest"] and M["query"] is not None:
                M["ingest"] = M["query"]
            else:
                M[role] = Embedder(dv[role])
        return M[role]


def reranker():
    with LOCK:
        if M["rerank"] is None:
            from sentence_transformers import CrossEncoder
            dev = devices()["query"]
            ce = CrossEncoder(RERANK_MODEL, device=dev, max_length=1024)
            if dev.startswith("cuda"):
                ce.model.half()
            M["rerank"] = ce
            log.info("loaded reranker %s on %s", RERANK_MODEL, dev)
        return M["rerank"]


def searcher():
    if M["searcher"] is None:
        M["searcher"] = R.Searcher(devices()["query"])
    return M["searcher"]


# ====================================================================== collections
def get_collection(name, create=False, **kw):
    with LOCK:
        if name in COLS:
            return COLS[name]
        try:
            col = R.Collection(COLS_DIR, name)
        except KeyError:
            if not create:
                raise K.HTTPError(404, "no collection named %r (create it, or ingest into it)" % name)
            col = R.Collection(COLS_DIR, name, model=EMBED_MODEL, create=True, **kw)
            log.info("created collection %s", name)
        COLS[name] = col
        return col


def create_collection(name, chunk_size=1200, chunk_overlap=150):
    if not R.NAME.match(str(name or "")):
        raise ValueError("collection names use letters, digits, - and _ (max 40)")
    if os.path.exists(os.path.join(COLS_DIR, name, "meta.json")):
        raise ValueError("collection %r already exists" % name)
    size, overlap = int(chunk_size or 1200), int(chunk_overlap or 0)
    if not 200 <= size <= 8000 or not 0 <= overlap <= size // 2:
        raise ValueError("chunk_size must be 200-8000 and chunk_overlap 0-chunk_size/2")
    return get_collection(name, create=True, chunk_size=size, chunk_overlap=overlap).stats()


def list_collections():
    out = []
    if os.path.isdir(COLS_DIR):
        for name in sorted(os.listdir(COLS_DIR)):
            if os.path.exists(os.path.join(COLS_DIR, name, "meta.json")):
                out.append(get_collection(name).stats())
    return out


def delete_collection(name):
    col = get_collection(name)
    with LOCK:
        if any(j["collection"] == name and j["state"] in ("queued", "running") for j in JOBS.values()):
            raise ValueError("an ingest job is using this collection")
        COLS.pop(name, None)
        if M["searcher"]:
            M["searcher"].forget(name)
        col.destroy()
    return {"deleted": name}


# ====================================================================== ingest jobs
def safe_input(path):
    full = os.path.realpath(str(path or ""))
    if not any(full == r or full.startswith(os.path.realpath(r) + os.sep) for r in INPUT_ROOTS):
        raise ValueError("paths must be under " + " or ".join(INPUT_ROOTS))
    if not os.path.exists(full):
        raise ValueError("no such file or folder: %s" % path)
    return full


def expand(items):
    """Folders become their supported files."""
    out = []
    for it in items:
        if it["type"] == "path" and os.path.isdir(it["path"]):
            for dp, dn, fn in os.walk(it["path"]):
                dn.sort()
                for f in sorted(fn):
                    if f.lower().endswith(P.SUPPORTED):
                        out.append({"type": "path", "path": os.path.join(dp, f), "name": f})
                        if len(out) > 20000:
                            raise ValueError("more than 20,000 files; ingest sub-folders separately")
        else:
            out.append(it)
    return out


def new_job(colname, items):
    if not items:
        raise ValueError("nothing to ingest: give urls, paths or texts")
    get_collection(colname, create=True)
    items = expand(items)
    job = {"id": uuid.uuid4().hex[:12], "collection": colname, "state": "queued", "created": time.time(),
           "total": len(items), "done": 0, "added": 0, "duplicates": 0, "failed": 0, "chunks": 0,
           "current": None, "results": [], "items": items}
    with WAKE:
        JOBS[job["id"]] = job
        QUEUE.append(job["id"])
        while len(JOBS) > 200:
            old = next(iter(JOBS))
            if JOBS[old]["state"] in ("queued", "running"):
                break
            JOBS.pop(old)
        WAKE.notify_all()
    log.info("job %s: %d items into %s", job["id"], len(items), colname)
    return job


def public(job, results=20):
    out = {k: v for k, v in job.items() if k not in ("items", "results")}
    out["results"] = job["results"][-results:] if results else []
    out["progress"] = round(job["done"] / job["total"], 3) if job["total"] else 1.0
    return out


def fetch_url(url, dest_dir):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (agentic-notebooks RAG ingest)"})
    with urllib.request.urlopen(req, timeout=60) as r:
        ctype = r.headers.get("Content-Type", "")
        name = os.path.basename(r.geturl().split("?")[0].rstrip("/")) or "page"
        kind = P.kind_of(name, ctype) or ("html" if "html" in ctype else None)
        if kind is None:
            raise ValueError("unsupported content type %r" % ctype)
        if P.kind_of(name) is None:
            name += {"html": ".html", "pdf": ".pdf", "docx": ".docx", "markdown": ".md"}.get(kind, ".txt")
        path = os.path.join(dest_dir, uuid.uuid4().hex[:8] + "_" + K.safe_name(name, 80))
        size = 0
        with open(path, "wb") as f:
            while True:
                data = r.read(1 << 20)
                if not data:
                    break
                size += len(data)
                if size > MAX_FETCH:
                    raise ValueError("larger than %d MB" % (MAX_FETCH >> 20))
                f.write(data)
    return path, name, ctype


def ingest_one(col, it):
    tmp = None
    try:
        if it["type"] == "text":
            text = str(it.get("text") or "")
            if not text.strip():
                raise ValueError("empty text")
            title = str(it.get("title") or text.strip().splitlines()[0][:80])
            blocks, _ = P._paras(text, markdown=True)
            parsed = {"title": title, "kind": "text", "blocks": blocks, "pages": None, "warnings": []}
            source, raw = it.get("source") or "text:" + title, text.encode("utf-8")
        else:
            if it["type"] == "url":
                os.makedirs(UPLOADS, exist_ok=True)
                tmp, name, ctype = fetch_url(it["url"], UPLOADS)
                path, source = tmp, it["url"]
            else:
                path, name, ctype = it["path"], it.get("name") or os.path.basename(it["path"]), ""
                source = it.get("source") or it["path"]
                if it["type"] == "upload":
                    tmp = path
            raw = open(path, "rb").read()
            parsed = P.parse_file(path, name, ctype)
        sha = hashlib.sha256(raw).hexdigest()
        dup = col.find_sha(sha)
        if dup:
            return {"source": source, "status": "duplicate", "document_id": dup}
        chunks = P.chunk_blocks(parsed["blocks"], col.meta["chunk_size"], col.meta["chunk_overlap"])
        if not chunks:
            return {"source": source, "status": "empty", "warnings": parsed["warnings"] or ["no text found"]}
        vecs = embedder("ingest").encode([P.embed_text(c, parsed["title"]) for c in chunks], DOC_PREFIX)
        doc_id = col.add_document({"source": source, "title": parsed["title"], "kind": parsed["kind"], "sha256": sha,
                                   "pages": parsed["pages"], "meta": {"warnings": parsed["warnings"]}}, chunks, vecs)
        return {"source": source, "status": "added", "document_id": doc_id, "title": parsed["title"],
                "chunks": len(chunks), "warnings": parsed["warnings"]}
    finally:
        if tmp and os.path.exists(tmp):
            os.remove(tmp)


def worker():
    while True:
        with WAKE:
            while not QUEUE:
                WAKE.wait(30)
            job = JOBS.get(QUEUE.pop(0))
        if not job:
            continue
        job.update(state="running", started=time.time())
        try:
            col = get_collection(job["collection"], create=True)
            for it in job["items"]:
                job["current"] = it.get("url") or it.get("name") or it.get("path") or it.get("title") or "text"
                try:
                    res = ingest_one(col, it)
                except Exception as e:
                    log.warning("ingest failed for %s: %s", job["current"], e)
                    res = {"source": job["current"], "status": "error", "error": "%s: %s" % (type(e).__name__, str(e)[:300])}
                job["results"].append(res)
                job["done"] += 1
                job["added"] += res["status"] == "added"
                job["duplicates"] += res["status"] == "duplicate"
                job["failed"] += res["status"] in ("error", "empty")
                job["chunks"] += res.get("chunks", 0) if res["status"] == "added" else 0
            job.update(state="done", current=None, finished=time.time())
        except Exception as e:
            log.error("job %s failed\n%s", job["id"], traceback.format_exc())
            job.update(state="error", error=str(e)[:500], current=None)
        for it in job["items"]:
            if it["type"] == "upload" and os.path.exists(it["path"]):
                os.remove(it["path"])
        log.info("job %s: %s, %d added, %d chunks", job["id"], job["state"], job["added"], job["chunks"])


def wait_job(jid, seconds):
    job = JOBS.get(jid)
    if not job:
        raise K.HTTPError(404, "unknown job %r (jobs are kept in memory until the server restarts)" % jid)
    t = time.time()
    while job["state"] in ("queued", "running") and time.time() - t < max(0, min(MAX_WAIT, float(seconds or 0))):
        time.sleep(1)
    return job


# ====================================================================== search
def search(colname, query, top_k=5, rerank=True, document_ids=None):
    col = get_collection(colname)
    query = str(query or "").strip()
    if not query:
        raise ValueError("query is empty")
    top_k = max(1, min(50, int(top_k or 5)))
    if col.meta.get("model") and col.meta["model"] != EMBED_MODEL:
        raise ValueError("this collection was embedded with %s but the server uses %s" % (col.meta["model"], EMBED_MODEL))
    t = time.time()
    q = embedder("query").encode([query], QUERY_PREFIX)[0]
    allowed = None
    if document_ids:
        ids = [int(d) for d in document_ids]
        allowed = [r[0] for r in col.db.execute("SELECT id FROM chunks WHERE deleted=0 AND doc_id IN (%s)"
                                                % ",".join("?" * len(ids)), ids)]
    hits = searcher().search(col, q, top_k * 4 if rerank else top_k, allowed)
    found = {c["chunk_id"]: c for c in (x["chunks"][0] for x in col.get_chunks([h[0] for h in hits]))}
    results = [dict(found[cid], score=round(s, 4)) for cid, s in hits if cid in found]
    if rerank and results:
        scores = reranker().predict([(query, r["text"]) for r in results], batch_size=16, show_progress_bar=False)
        for r, s in zip(results, scores):
            r["rerank_score"] = round(float(s), 4)
        results.sort(key=lambda r: -r["rerank_score"])
    return {"query": query, "collection": colname, "results": results[:top_k], "ms": round((time.time() - t) * 1000)}


def list_inputs():
    out = []
    for root in INPUT_ROOTS:
        for dp, dn, fn in os.walk(root):
            if os.path.realpath(dp).startswith(os.path.realpath(WORK)):
                dn[:] = []
                continue
            dn.sort()
            files = [f for f in sorted(fn) if f.lower().endswith(P.SUPPORTED)]
            if files:
                out.append({"folder": dp, "files": len(files), "examples": files[:5]})
            if len(out) >= 300:
                return out
    return out


def info():
    dv = devices()
    return {"embed_model": EMBED_MODEL, "rerank_model": RERANK_MODEL, "devices": dv, "gpus": K.gpu_info(),
            "loaded": {k: M[k] is not None for k in ("ingest", "query", "rerank")},
            "jobs": {s: sum(1 for j in JOBS.values() if j["state"] == s) for s in ("queued", "running", "done", "error")},
            "supported": sorted(P.KINDS), "input_roots": INPUT_ROOTS}


# ====================================================================== app
def build_app():
    app = K.App("RAG Ingest Pipeline", password=os.environ.get("APP_PASSWORD"), log=log, max_body=1 << 30,
                instructions=(
                    "Document vectorizer and semantic search. ingest() queues files/folders under /kaggle/input "
                    "(see list_input_files), URLs or raw texts into a collection (created on first use) and returns "
                    "a job; poll job_status. search() returns the best chunks with title, source, pages and scores; "
                    "use get_chunks with context>0 to read around a hit. export_collection gives a zip "
                    "(chunks.jsonl + embeddings.npy) for any vector database. Uploads go through REST: "
                    "POST /api/collections/<name>/upload?name=<file> with the raw file body."))
    app.page("/", os.path.join(HERE, "rag_ui.html"))
    app.static("/static/", HERE)

    @app.route("GET", "/api/info")
    def _info(req):
        return info()

    @app.route("GET", "/api/collections")
    def _cols(req):
        return {"collections": list_collections()}

    @app.route("POST", "/api/collections")
    def _create(req):
        b = req.json()
        return create_collection(b.get("name"), b.get("chunk_size", 1200), b.get("chunk_overlap", 150))

    @app.route("POST", r"/api/collections/(?P<c>[\w-]+)/delete")
    def _delcol(req):
        return delete_collection(req.params["c"])

    @app.route("GET", r"/api/collections/(?P<c>[\w-]+)/documents")
    def _docs(req):
        return get_collection(req.params["c"]).documents(req.arg("offset", 0, int), min(200, req.arg("limit", 50, int)),
                                                     req.arg("q", "") or "")

    @app.route("GET", r"/api/collections/(?P<c>[\w-]+)/documents/(?P<d>\d+)")
    def _doc(req):
        return get_collection(req.params["c"]).document(req.params["d"])

    @app.route("POST", r"/api/collections/(?P<c>[\w-]+)/documents/(?P<d>\d+)/delete")
    def _deldoc(req):
        return get_collection(req.params["c"]).delete_document(req.params["d"])

    @app.route("POST", r"/api/collections/(?P<c>[\w-]+)/upload")
    def _upload(req):
        name = K.safe_name(req.arg("name", "upload.txt"), 100) or "upload.txt"
        if P.kind_of(name) is None:
            raise ValueError("unsupported file type %s; supported: %s" % (name, " ".join(sorted(P.KINDS))))
        os.makedirs(UPLOADS, exist_ok=True)
        path = os.path.join(UPLOADS, uuid.uuid4().hex[:8] + "_" + name)
        req.save_body(path, 1 << 30)
        try:
            job = new_job(req.params["c"], [{"type": "upload", "path": path, "name": name, "source": "upload:" + name}])
        except BaseException:
            os.remove(path)
            raise
        return public(job)

    @app.route("POST", r"/api/collections/(?P<c>[\w-]+)/ingest")
    def _ingest(req):
        b = req.json()
        return public(t_ingest(req.params["c"], b.get("urls"), b.get("paths"), b.get("texts")))

    @app.route("POST", r"/api/collections/(?P<c>[\w-]+)/search")
    def _search(req):
        b = req.json()
        return search(req.params["c"], b.get("query"), b.get("top_k", 5), b.get("rerank", True), b.get("document_ids"))

    @app.route("GET", r"/api/collections/(?P<c>[\w-]+)/export")
    def _export(req):
        col = get_collection(req.params["c"])
        return K.ZipResponse(col.export_files(), "%s-export.zip" % col.name)

    @app.route("GET", "/api/jobs")
    def _jobs(req):
        return {"jobs": [public(j, 0) for j in reversed(list(JOBS.values()))]}

    @app.route("GET", r"/api/jobs/(?P<j>\w+)")
    def _job(req):
        return public(wait_job(req.params["j"], req.arg("wait", 0, float)), 500)

    @app.route("GET", "/api/sources")
    def _sources(req):
        return {"folders": list_inputs()}

    # ---------------------------------------------------------------- MCP tools
    @app.tool("list_collections", "Collections with document and chunk counts, embedding model and chunk settings.")
    def t_cols():
        return list_collections()

    @app.tool("create_collection", "Create an empty collection. ingest() also creates one with default settings.", {
        "name": {"type": "string"}, "chunk_size": {"type": "integer", "default": 1200, "description": "characters"},
        "chunk_overlap": {"type": "integer", "default": 150}}, ["name"])
    def t_create(name, chunk_size=1200, chunk_overlap=150):
        return create_collection(name, chunk_size, chunk_overlap)

    @app.tool("ingest", "Add documents to a collection: URLs (web pages, PDFs, DOCX...), file or folder paths under "
              "/kaggle/input, and/or raw texts. Duplicates are skipped. Returns a job; poll job_status.", {
                  "collection": {"type": "string"},
                  "urls": {"type": "array", "items": {"type": "string"}},
                  "paths": {"type": "array", "items": {"type": "string"}},
                  "texts": {"type": "array", "items": {"type": "object", "properties": {
                      "title": {"type": "string"}, "text": {"type": "string"}, "source": {"type": "string"}}}},
                  "wait_seconds": {"type": "number", "default": 0}}, ["collection"])
    def t_ingest(collection, urls=None, paths=None, texts=None, wait_seconds=0):
        colname = collection
        if not R.NAME.match(str(colname or "")):
            raise ValueError("collection names use letters, digits, - and _ (max 40)")
        items = []
        for u in urls or []:
            if not re.match(r"^https?://", str(u)):
                raise ValueError("urls must start with http:// or https://: %s" % u)
            items.append({"type": "url", "url": str(u)})
        for pth in paths or []:
            items.append({"type": "path", "path": safe_input(pth), "name": os.path.basename(str(pth))})
        for t in texts or []:
            if not isinstance(t, dict):
                t = {"text": str(t)}
            items.append({"type": "text", "text": t.get("text", ""), "title": t.get("title"), "source": t.get("source")})
        job = new_job(colname, items)
        return public(wait_job(job["id"], wait_seconds)) if wait_seconds else public(job)

    @app.tool("job_status", "Progress and per-item results of an ingest job.", {
        "job_id": {"type": "string"}, "wait_seconds": {"type": "number", "default": 0}}, ["job_id"])
    def t_job(job_id, wait_seconds=0):
        return public(wait_job(job_id, wait_seconds), 100)

    @app.tool("search", "Semantic search in a collection. Returns chunks with text, title, source, pages, heading and "
              "scores (rerank_score when rerank is on).", {
                  "collection": {"type": "string"}, "query": {"type": "string"},
                  "top_k": {"type": "integer", "default": 5, "maximum": 50},
                  "rerank": {"type": "boolean", "default": True},
                  "document_ids": {"type": "array", "items": {"type": "integer"}, "description": "only these documents"}},
              ["collection", "query"])
    def t_search(collection, query, top_k=5, rerank=True, document_ids=None):
        return search(collection, query, top_k, rerank, document_ids)

    @app.tool("get_chunks", "Read chunks by id, with `context` neighbouring chunks on each side.", {
        "collection": {"type": "string"}, "chunk_ids": {"type": "array", "items": {"type": "integer"}},
        "context": {"type": "integer", "default": 1, "maximum": 5}}, ["collection", "chunk_ids"])
    def t_chunks(collection, chunk_ids, context=1):
        return get_collection(collection).get_chunks(chunk_ids, context)

    @app.tool("list_documents", "Documents in a collection, newest first.", {
        "collection": {"type": "string"}, "q": {"type": "string", "description": "filter by title or source"},
        "offset": {"type": "integer", "default": 0}, "limit": {"type": "integer", "default": 50}}, ["collection"])
    def t_docs(collection, q="", offset=0, limit=50):
        return get_collection(collection).documents(offset, min(200, int(limit)), q or "")

    @app.tool("delete_document", "Remove a document (and its chunks) from search.", {
        "collection": {"type": "string"}, "document_id": {"type": "integer"}}, ["collection", "document_id"])
    def t_deldoc(collection, document_id):
        return get_collection(collection).delete_document(document_id)

    @app.tool("export_collection", "Where to download a collection as a zip: chunks.jsonl, embeddings.npy (float16, "
              "L2-normalised), documents.jsonl, manifest.json.", {"collection": {"type": "string"}}, ["collection"])
    def t_export(collection):
        st = get_collection(collection).stats()
        return {"download": "/api/collections/%s/export" % collection, "documents": st["documents"],
                "chunks": st["chunks"], "note": "GET it with the same password (Authorization: Bearer <password>)"}

    @app.tool("list_input_files", "Folders under /kaggle/input with supported documents, to pass to ingest as paths.")
    def t_inputs():
        return list_inputs()

    @app.tool("server_status", "Models, devices, GPU memory and job counts.")
    def t_status():
        return info()

    return app


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    os.makedirs(COLS_DIR, exist_ok=True)
    shutil.rmtree(UPLOADS, ignore_errors=True)       # leftovers from a previous run
    threading.Thread(target=worker, daemon=True).start()
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'rag_server.py'))

In [ ]:
# Page
# Writes rag_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/rag_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'rag_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>RAG Ingest</title>
<link rel="stylesheet" href="/static/kit.css">
<style>
.layout{display:grid;grid-template-columns:260px 1fr;gap:16px;align-items:start}
@media (max-width:860px){.layout{grid-template-columns:1fr}}
.colitem{padding:8px 10px;border:1px solid var(--line);border-radius:8px;cursor:pointer;background:var(--panel2);margin-bottom:6px}
.colitem.on{border-color:var(--accent)}
.hit{border:1px solid var(--line);border-radius:8px;padding:10px 12px;margin-bottom:10px}
.hit .meta{font-size:12px;color:var(--muted);display:flex;gap:10px;flex-wrap:wrap;margin-bottom:6px}
.hit .tx{white-space:pre-wrap;font-size:13.5px;max-height:220px;overflow:auto}
textarea.small{min-height:70px}
</style></head><body>
<header class="top"><h1>RAG Ingest Pipeline</h1><span class="sub" id="models"></span><div class="right small muted" id="status"></div></header>
<main><div class="layout">
  <div class="card"><h2>Collections</h2><div id="cols"></div>
    <div class="col" style="margin-top:10px"><input id="newname" placeholder="new-collection">
      <div class="row"><label>Chunk chars<input id="csize" type="number" value="1200" style="width:90px"></label><label>Overlap<input id="cover" type="number" value="150" style="width:70px"></label></div>
      <button id="create">Create</button></div></div>
  <div>
    <div class="card" id="nocol"><div class="empty">Create or pick a collection.</div></div>
    <div id="colview" hidden>
      <div class="card"><div class="row"><h2 style="margin:0" id="ctitle"></h2><span class="small muted" id="cstats"></span>
        <span style="margin-left:auto" class="row"><a class="btn" id="export">Export zip</a><button class="danger" id="delcol">Delete</button></span></div></div>
      <div class="card"><h2>Search</h2>
        <form id="sf" class="row"><input id="query" placeholder="Ask something…" style="flex:1"><label class="inline"><input type="checkbox" id="rerank" checked> Rerank</label>
          <select id="topk"><option>5</option><option selected>8</option><option>15</option><option>30</option></select><button class="primary">Search</button></form>
        <div id="hits" style="margin-top:12px"></div></div>
      <div class="card"><h2>Add documents</h2>
        <div class="grid">
          <div class="col"><div class="drop" id="drop">Drop files (PDF, DOCX, HTML, Markdown, text, code, notebooks)</div><input type="file" id="file" multiple hidden>
            <div class="small muted">Up to 100 MB each through Cloudflare. For big corpora attach a Kaggle dataset and ingest its folder.</div></div>
          <div class="col"><textarea class="small" id="urls" placeholder="URLs, one per line"></textarea>
            <div class="row"><select id="folder" style="flex:1"><option value="">Folder or file under /kaggle/input…</option></select></div>
            <button id="addsrc">Ingest URLs / folder</button></div>
          <div class="col"><input id="ttitle" placeholder="Title"><textarea class="small" id="ttext" placeholder="…or paste text"></textarea><button id="addtext">Ingest text</button></div>
        </div>
        <div id="jobs" style="margin-top:12px"></div></div>
      <div class="card"><div class="row"><h2 style="margin:0">Documents</h2><input id="dq" placeholder="Filter" style="margin-left:auto"></div>
        <table><thead><tr><th>Title</th><th>Source</th><th>Chunks</th><th></th></tr></thead><tbody id="docs"></tbody></table></div>
    </div>
  </div>
</div>
<div class="card"><h2>For agents</h2><p class="small muted">MCP tools: list_collections, create_collection, ingest, job_status, search, get_chunks, list_documents, delete_document, export_collection, list_input_files, server_status.</p><pre class="log" id="mcp"></pre></div>
</main>
<script src="/static/kit.js"></script>
<script>
let cur = null;
$("#mcp").textContent = mcpHelp("rag");

async function loadInfo() {
  const i = await api("/api/info");
  $("#models").textContent = i.embed_model + " · rerank " + i.rerank_model;
  $("#status").textContent = (i.gpus.length ? i.gpus.length + " GPU" : "CPU") + " · jobs " + i.jobs.running + " running, " + i.jobs.queued + " queued";
}
async function loadCols() {
  const {collections} = await api("/api/collections");
  $("#cols").replaceChildren(...collections.map(c => el("div", {class: "colitem" + (c.name === cur ? " on" : ""), onclick: () => pick(c.name)},
    el("b", {}, c.name), el("div", {class: "small muted"}, c.documents + " docs · " + c.chunks.toLocaleString() + " chunks"))));
  if (!collections.length) $("#cols").replaceChildren(el("div", {class: "small muted"}, "None yet."));
  const c = collections.find(c => c.name === cur);
  if (c) $("#cstats").textContent = c.documents + " documents · " + c.chunks.toLocaleString() + " chunks · " + fmtBytes(c.disk_bytes) + " · " + c.chunk_size + "/" + c.chunk_overlap + " chars · " + c.model;
}
function pick(name) { cur = name; $("#nocol").hidden = true; $("#colview").hidden = false; $("#ctitle").textContent = name;
  $("#export").href = "/api/collections/" + name + "/export"; $("#hits").replaceChildren(); loadCols(); loadDocs(); }
$("#create").onclick = async () => { const name = $("#newname").value.trim(); await guard(() => api("/api/collections", {json: {name, chunk_size: +$("#csize").value, chunk_overlap: +$("#cover").value}}), "Created"); pick(name); };
$("#delcol").onclick = async () => { if (!confirm("Delete collection " + cur + " and everything in it?")) return; await guard(() => api("/api/collections/" + cur + "/delete", {json: {}})); cur = null; $("#colview").hidden = true; $("#nocol").hidden = false; loadCols(); };

async function loadDocs() {
  if (!cur) return;
  const d = await api("/api/collections/" + cur + "/documents?limit=100&q=" + encodeURIComponent($("#dq").value));
  $("#docs").replaceChildren(...d.documents.map(x => el("tr", {}, el("td", {}, x.title, (x.meta.warnings || []).length ? el("div", {class: "small muted"}, x.meta.warnings.join("; ")) : null),
    el("td", {class: "small muted", style: {wordBreak: "break-all"}}, x.source), el("td", {}, x.chunks),
    el("td", {}, el("button", {class: "danger", onclick: async () => { await guard(() => api("/api/collections/" + cur + "/documents/" + x.id + "/delete", {json: {}})); loadDocs(); loadCols(); }}, "×")))));
  if (!d.documents.length) $("#docs").replaceChildren(el("tr", {}, el("td", {colspan: 4, class: "empty"}, "No documents yet.")));
}
$("#dq").oninput = loadDocs;

const watching = new Set();
async function track(job) { watching.add(job.id); renderJobs(); }
async function renderJobs() {
  const rows = [];
  for (const id of [...watching]) {
    const j = await api("/api/jobs/" + id);
    rows.push(el("div", {class: "col", style: {marginBottom: "8px"}},
      el("div", {class: "row small"}, badge(j.state), j.done + "/" + j.total + " files", j.added + " added", j.duplicates ? j.duplicates + " duplicates" : "", j.failed ? j.failed + " failed" : "", j.chunks + " chunks", j.current ? el("span", {class: "muted"}, "· " + j.current) : ""),
      progressBar(j.progress),
      ...j.results.filter(r => r.status === "error" || r.status === "empty").slice(-5).map(r => el("div", {class: "small", style: {color: "var(--bad)"}}, r.source + ": " + (r.error || (r.warnings || []).join("; "))))));
    if (j.state === "done" || j.state === "error") { setTimeout(() => { watching.delete(id); }, 20000); loadDocs(); loadCols(); }
  }
  $("#jobs").replaceChildren(...rows);
}
dropZone($("#drop"), $("#file"), async files => {
  for (const f of files) {
    const j = await guard(() => api("/api/collections/" + cur + "/upload?name=" + encodeURIComponent(f.name), {body: f}));
    track(j);
  }
});
$("#addsrc").onclick = async () => {
  const urls = $("#urls").value.split("\n").map(s => s.trim()).filter(Boolean), paths = $("#folder").value ? [$("#folder").value] : [];
  const j = await guard(() => api("/api/collections/" + cur + "/ingest", {json: {urls, paths}}), "Queued"); $("#urls").value = ""; track(j);
};
$("#addtext").onclick = async () => {
  const j = await guard(() => api("/api/collections/" + cur + "/ingest", {json: {texts: [{title: $("#ttitle").value, text: $("#ttext").value}]}}), "Queued");
  $("#ttext").value = ""; $("#ttitle").value = ""; track(j);
};
$("#sf").addEventListener("submit", async e => {
  e.preventDefault();
  $("#hits").replaceChildren(el("div", {class: "muted small"}, "Searching… (the first search loads the models)"));
  const r = await guard(() => api("/api/collections/" + cur + "/search", {json: {query: $("#query").value, top_k: +$("#topk").value, rerank: $("#rerank").checked}}));
  $("#hits").replaceChildren(el("div", {class: "small muted", style: {marginBottom: "8px"}}, r.results.length + " results in " + r.ms + " ms"),
    ...r.results.map(h => el("div", {class: "hit"}, el("div", {class: "meta"}, el("b", {}, h.title), h.heading ? "§ " + h.heading : "",
      h.page_start ? "p. " + h.page_start + (h.page_end && h.page_end !== h.page_start ? "–" + h.page_end : "") : "",
      "score " + h.score + (h.rerank_score != null ? " · rerank " + h.rerank_score : ""), "chunk " + h.chunk_id),
      el("div", {class: "tx"}, h.text), el("div", {class: "small muted", style: {marginTop: "6px", wordBreak: "break-all"}}, h.source))));
});
(async () => {
  try { const {folders} = await api("/api/sources");
    folders.forEach(f => $("#folder").append(el("option", {value: f.folder}, f.folder.replace("/kaggle/input/", "") + " (" + f.files + " files)"))); } catch (e) {}
})();
every(5000, loadInfo);
every(8000, loadCols);
every(2000, () => watching.size ? renderJobs() : null);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'rag_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/rag_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install and download the models (quiet; about 4 minutes; re-running skips finished steps)
import subprocess
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

os.makedirs(LOG_DIR, exist_ok=True)
print("sentence-transformers, pymupdf, python-docx...")
pip_install("sentence-transformers", "pymupdf", "python-docx")
K.ensure_cloudflared()
from huggingface_hub import snapshot_download
for m in (EMBED_MODEL, RERANK_MODEL):
    print("Downloading", m, "...")
    snapshot_download(m, token=K.kaggle_secret("HF_TOKEN"),
                      ignore_patterns=["onnx/*", "*.onnx", "*.msgpack", "*.h5", "*.ot", "openvino/*"])
gpus = K.gpu_info()
print("Install complete. GPUs:", ", ".join("%d: %s" % (g["index"], g["name"]) for g in gpus) or "none (CPU is slow)")

In [ ]:
# 5. Start the pipeline on port 7860 (background process). Log: /kaggle/working/logs/rag.log
PASSWORD = K.ui_password("RAG_UI_PASSWORD")
env = dict(os.environ, APP_PASSWORD=PASSWORD, WORK_DIR=WORK_DIR, APP_LOG=LOG_DIR + "/rag.log", PYTHONUNBUFFERED="1",
           EMBED_MODEL=EMBED_MODEL, RERANK_MODEL=RERANK_MODEL, QUERY_PREFIX=QUERY_PREFIX, DOC_PREFIX=DOC_PREFIX,
           EMBED_BATCH=str(EMBED_BATCH), TOKENIZERS_PARALLELISM="false")
K.spawn("rag-pipeline", [sys.executable, os.path.join(APP_DIR, "rag_server.py"), "--port", str(PORT)],
        LOG_DIR + "/rag.log", env=env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=60, name="rag-pipeline"):
    print(K.tail(LOG_DIR + "/rag.log", 40))
    raise RuntimeError("The pipeline did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print("Models load on first use (about 30 s).")

In [ ]:
# 6. Publish through your Cloudflare Tunnel (token from the RAG_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("RAG_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 7. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def jobs_line():
    j = K.api_get(PORT, "/api/info", PASSWORD)["jobs"]
    return "ingest jobs: %d running, %d queued" % (j["running"], j["queued"])

K.keep_alive(PORT, ["rag-pipeline", "cloudflared"], extra=jobs_line)